# Lección 03: IA en Biotecnologia

Matriz de confusión, metricas de clasificación y datos estructurales del PDB.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

## Matriz de confusión desde cero (BCW + KNN)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusion:')
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}')

## Precisión, recall y F1-score

In [ ]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f'Manual  -> accuracy: {accuracy:.3f}, precision: {precision:.3f}, recall: {recall:.3f}, f1: {f1:.3f}')
print(f'sklearn -> accuracy: {accuracy_score(y_test, y_pred):.3f}, '
      f'precision: {precision_score(y_test, y_pred):.3f}, '
      f'recall: {recall_score(y_test, y_pred):.3f}, '
      f'f1: {f1_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

## Fallback: coordenadas Cα de 1UBQ

In [ ]:
FALLBACK_CA_CSV = """
residue_number,x,y,z
1,26.266,25.413,2.842
2,26.850,29.021,3.898
3,26.235,30.058,7.497
4,26.772,33.436,9.197
5,28.605,33.965,12.503
6,27.691,37.315,14.143
7,30.225,38.643,16.662
8,29.607,41.180,19.467
9,31.422,43.940,17.553
10,28.978,43.960,14.678
11,31.191,42.012,12.331
12,29.542,39.020,10.653
13,31.720,36.289,9.176
14,30.505,33.884,6.512
15,31.677,30.275,6.639
16,31.220,27.341,4.275
17,30.288,24.245,6.193
18,28.468,20.940,5.980
19,25.829,19.825,8.494
20,28.054,16.835,9.210
21,30.796,19.083,10.566
22,31.398,19.064,14.286
23,31.288,22.201,16.417
24,35.031,21.722,17.069
25,35.590,21.945,13.302
26,33.533,25.097,12.978
27,35.596,26.715,15.736
28,38.794,25.761,13.880
29,37.471,27.391,10.668
30,36.731,30.570,12.645
31,40.269,30.508,14.115
32,41.718,30.022,10.643
33,39.808,32.994,9.233
34,39.676,35.547,12.072
35,42.345,34.269,14.431
36,40.226,33.716,17.509
37,41.461,30.751,19.594
38,38.817,28.020,19.889
39,39.063,28.063,23.695
40,37.738,31.637,23.712
41,34.738,30.875,21.473
42,31.200,30.329,22.780
43,28.762,29.573,19.906
44,25.034,30.170,20.401
45,22.126,29.062,18.183
46,18.443,29.143,19.083
47,19.399,29.894,22.655
48,21.550,26.796,23.133
49,25.349,26.872,23.643
50,26.826,24.521,21.012
51,29.015,21.657,22.288
52,32.262,20.670,20.514
53,31.568,16.962,19.825
54,28.108,17.439,18.276
55,27.574,18.192,14.563
56,25.594,21.109,13.072
57,22.924,18.583,12.025
58,22.418,17.638,15.693
59,21.079,21.149,16.251
60,19.065,21.352,12.999
61,21.184,24.263,11.690
62,20.081,24.773,8.033
63,21.656,26.847,5.240
64,21.907,30.563,5.881
65,21.419,30.253,9.620
66,23.212,32.762,11.891
67,25.149,31.609,14.980
68,26.179,34.127,17.650
69,29.801,34.145,18.829
70,30.479,35.369,22.374
71,34.145,35.472,23.481
72,35.161,34.174,26.896
73,38.668,35.502,27.680
74,40.873,33.802,30.253
75,41.845,36.550,32.686
76,40.373,39.813,33.944
"""

## Descarga de la estructura del PDB

In [ ]:
pdb_id = '1UBQ'
structure = None

try:
    from Bio.PDB import PDBList, PDBParser

    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_file)
    print(f'Estructura {pdb_id} cargada con Biopython')
except Exception as e:
    print(f'Biopython no disponible o fallo: {e}')
    print('Usando fallback CSV embebido...')
    structure = FALLBACK_CA_CSV

## Parseo de coordenadas Cα

In [ ]:
def parse_ca_from_text(pdb_text):
    coords = []
    for line in pdb_text.splitlines():
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            resi = int(line[22:26])
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            coords.append((resi, x, y, z))
    return coords

if isinstance(structure, str):
    ca_coords = parse_ca_from_text(structure)
else:
    ca_coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.has_id('CA'):
                    atom = residue['CA']
                    ca_coords.append((residue.id[1], atom.coord[0], atom.coord[1], atom.coord[2]))

print(f'Residuos Cα: {len(ca_coords)}')
print('Primeros 3:', ca_coords[:3])

## Distancias entre Cα

In [ ]:
first = np.array(ca_coords[0][1:])
last = np.array(ca_coords[-1][1:])
end_to_end = np.linalg.norm(first - last)
print(f'Distancia Cα(1) -> Cα({ca_coords[-1][0]}): {end_to_end:.2f} Å')

# Matriz de distancias para los primeros 10 residuos
indices = np.arange(10)
coord_matrix = np.array([c[1:] for c in ca_coords[:10]])
dist_matrix = np.sqrt(((coord_matrix[:, None, :] - coord_matrix[None, :, :]) ** 2).sum(axis=2))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dist_matrix, cmap='viridis')
ax.set_xticks(indices)
ax.set_yticks(indices)
ax.set_xlabel('Residuo')
ax.set_ylabel('Residuo')
ax.set_title('Matriz de distancias Cα (primeros 10 residuos)')
fig.colorbar(im, ax=ax, label='Distancia (Å)')
plt.tight_layout()
plt.show()

## Resumen del pipeline

In [ ]:
pipeline_summary = pd.DataFrame([
    {'Etapa': 'Datos', 'Descripcion': 'Cargar BCW (569 muestras, 30 features)', 'Herramienta': 'sklearn.datasets'},
    {'Etapa': 'Split', 'Descripcion': 'Separar 70% train / 30% test estratificado', 'Herramienta': 'train_test_split'},
    {'Etapa': 'Modelo', 'Descripcion': 'Entrenar KNN con k=5', 'Herramienta': 'KNeighborsClassifier'},
    {'Etapa': 'Evaluacion', 'Descripcion': 'Matriz de confusion + metricas', 'Herramienta': 'sklearn.metrics'},
    {'Etapa': 'Estructural', 'Descripcion': 'Cargar Cα de 1UBQ', 'Herramienta': 'Bio.PDB / fallback CSV'},
])

print(pipeline_summary.to_string(index=False))
print()
print(f'Accuracy : {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall   : {recall:.3f}')
print(f'F1-score : {f1:.3f}')